# Clase 09 — Ejercicio intergrador Clínica MediCare

> Notebook de práctica. Ejecutá las celdas a medida que avanzás en el artículo.

La clínica MediCare lleva varios años registrando turnos médicos en un sistema interno. El sistema fue cambiando de versión con el tiempo, y los datos se fueron exportando a una base de datos SQLite sin demasiado control de calidad.

El director médico necesita un informe con algunas preguntas concretas sobre el funcionamiento de la clínica, pero antes de poder responderlas, alguien tiene que hacer la limpieza. Ese trabajo es el de ustedes, no se les dice qué problemas hay en los datos. Parte del ejercicio es encontrarlos.

> El archivo de la base de datos `medicare.db` ya está disponible para que lo uses en este ejercicio.

### 1) Exploración: ¿qué tiene esta base de datos?

Antes de tocar nada, el primer trabajo es entender qué hay adentro. Conectense a `medicare.db` y respondan estas preguntas.

1. ¿Qué tablas tiene la base de datos?
2. ¿Qué columnas y tipos tiene cada tabla?
3. ¿Cuántas filas tiene cada tabla?

In [5]:
# tu codigo aca
import sqlite3
import pandas as pd

conexion = sqlite3.connect('medicare.db')

df = pd.read_sql(
    """
    SELECT *
    FROM sqlite_master
    """,
    conexion
) 

#print (df) #1 ¿Qué tablas tiene la base de datos?

df = pd.read_sql(
    """
    PRAGMA table_info(pacientes)

    """,
    conexion
)

#print (df) #2 ¿Qué columnas y tipos tiene cada tabla?

df = pd.read_sql(
    """
    PRAGMA table_info(medicos)

    """,
    conexion
)

#print (df) #2

df = pd.read_sql(
    """
    PRAGMA table_info(turnos)

    """,
    conexion
)

#print (df) #2

df = pd.read_sql(
    """
    SELECT COUNT (*)
    FROM pacientes
    """,
    conexion
)

print(df)#3 ¿Cuántas filas tiene cada tabla?


   COUNT (*)
0         66


**Documenten sus hallazgos.** Antes de seguir al Paso 2, escriban en comentarios qué encontraron.
La BDD tiene 3 tablas , para saber la estructura de cada tabla se uso pragma (table_info() ) por ultimo se conto la cantidad de registros de cada tabla usando COUNT(*)

### 2) Diagnóstico: ¿qué problemas tienen los datos?

Traigan las tres tablas a DataFrames y hagan un diagnóstico completo de cada una. Para cada tabla respondan:

- ¿Cuántos nulos hay por columna? ¿Qué porcentaje representan?
- ¿Hay filas duplicadas? ¿Cuáles?
- ¿Hay columnas con tipos incorrectos que van a impedir operar con los datos?
- ¿Hay valores que parecen errores aunque no sean nulos? (Pista: revisen la columna `costo` en `turnos`)


In [6]:
# tu codigo aquí

df_pacientes = pd.read_sql(
    """
    SELECT *
    FROM pacientes
    """,
    conexion
)

diagnostico_nulos_pacientes = pd.DataFrame({
    "Cantidad de nulos PACIENTES": df_pacientes.isnull().sum(),
    "Porcentaje de nulos PACIENTES": (df_pacientes.isnull().sum() / len(df_pacientes)) * 100
})

print(diagnostico_nulos_pacientes)

print("Cantidad de filas duplicadas exactas:")
print(df_pacientes.duplicated().sum())

print("Duplicados por DNI:")
print(df_pacientes.duplicated(subset=["dni"]).sum())

print("Tipos de datos:")
print(df_pacientes.dtypes)


# MEDICOS

df_medicos = pd.read_sql(
    """
    SELECT *
    FROM medicos
    """,
    conexion
)

diagnostico_nulos_medicos = pd.DataFrame({
    "Cantidad de nulos MEDICOS": df_medicos.isnull().sum(),
    "Porcentaje de nulos MEDICOS": (df_medicos.isnull().sum() / len(df_medicos)) * 100
})

print(diagnostico_nulos_medicos)

print("Cantidad de filas duplicadas exactas:")
print(df_medicos.duplicated().sum())

print("Duplicados por nombre:")
print(df_medicos.duplicated(subset=["nombre"]).sum())

print("Tipos de datos:")
print(df_medicos.dtypes)


# TURNOS

df_turnos = pd.read_sql(
    """
    SELECT *
    FROM turnos
    """,
    conexion
)

diagnostico_nulos_turnos = pd.DataFrame({
    "Cantidad de nulos TURNOS": df_turnos.isnull().sum(),
    "Porcentaje de nulos TURNOS": (df_turnos.isnull().sum() / len(df_turnos)) * 100
})

print(diagnostico_nulos_turnos)

print("Cantidad de filas duplicadas exactas:")
print(df_turnos.duplicated().sum())

print("Tipos de datos:")
print(df_turnos.dtypes)


# Valores problemáticos en costo

print("Valores de costo:")
print(df_turnos["costo"].unique())

                  Cantidad de nulos PACIENTES  Porcentaje de nulos PACIENTES
id                                          0                       0.000000
nombre                                      0                       0.000000
dni                                         7                      10.606061
fecha_nacimiento                           11                      16.666667
ciudad                                      8                      12.121212
obra_social                                12                      18.181818
Cantidad de filas duplicadas exactas:
0
Duplicados por DNI:
14
Tipos de datos:
id                  int64
nombre                str
dni                   str
fecha_nacimiento      str
ciudad                str
obra_social           str
dtype: object
              Cantidad de nulos MEDICOS  Porcentaje de nulos MEDICOS
id                                    0                          0.0
nombre                                0                          0.0
espec

**Antes de seguir al Paso 3, escriban una lista con todos los problemas que encontraron, uno por línea.** No empiecen a limpiar todavía.
 Problemas encontrados:
 - Hay valores nulos en varias columnas de la tabla pacientes.
 - La columna fecha_nacimiento está almacenada como texto.
 - La columna costo de la tabla turnos está almacenada como texto.
 - La columna costo contiene valores negativos, "N/A", cadenas vacías y valores nulos.

### 3) Decisiones: ¿qué hacemos con cada problema?

Este es el paso más importante del ejercicio. Para cada problema que encontraron en el Paso 2, decidan qué van a hacer y por qué. No hay una respuesta única correcta — lo que importa es que la decisión sea coherente con el análisis que necesitan hacer después.

Completen esta tabla antes de escribir código:

| Tabla | Columna | Problema | Decisión | Justificación |
|-------|---------|----------|----------|---------------|
| pacientes | dni | 7 nulo | mantener | No se pueden completar |
| pacientes | fecha de nacimiento | 11 nulo y tipo str | convertir | debe ser una fecha |
| pacientes | ciudad | 8 nulos | mantener | No se conoce el valor |
| pacientes | obra social | 12 nulo | mantener | No se puede asumir |
| medicos | matricula | 2 nulo | mantener | No se pueden completar |
| medicos | activo | 4 nulo | mantener | No se conoce el estado |
| turnos | costos | tipo str | convertir | debe ser numerico |
| turnos | costos | Valores inválidos (N/A, "", negativos) | reemplazar por NaN | Facilita el análisis |






### 4) Limpieza: aplicar las decisiones

Ahora sí, implementen las decisiones del Paso 3. Recuerden:

- Primero eliminen duplicados
- Después manejen nulos
- Después conviertan tipos

In [7]:
#PACIENTE
df_pacientes_limpio = df_pacientes.copy()#limpieza
df_pacientes_limpio = df_pacientes_limpio.drop_duplicates()#duplocados

df_pacientes_limpio["fecha_nacimiento"] = pd.to_datetime(# Convertir datatime
    df_pacientes_limpio["fecha_nacimiento"],
    errors="coerce")

#MEDICOS
df_medicos_limpio = df_medicos.copy()
df_medicos_limpio = df_medicos_limpio.drop_duplicates()

#TURNO
df_turnos_limpio = df_turnos.copy()
df_turnos_limpio = df_turnos_limpio.drop_duplicates()

df_turnos_limpio["costo"] = df_turnos_limpio["costo"].replace(#reemplaza valores invalidos
    ["N/A", "", "-500", "-1000", "-3500"],
    pd.NA
)

df_turnos_limpio["costo"] = pd.to_numeric( #convertir numero
    df_turnos_limpio["costo"],
    errors="coerce"
)

> **Por qué `.copy()`:** cuando hacen `df_pacientes_limpio = df_pacientes`, no están creando una copia — están apuntando al mismo objeto. Cualquier cambio en `df_pacientes_limpio` modifica también `df_pacientes`. `.copy()` crea una copia independiente, así conservan los datos originales para comparar.

Al terminar, impriman un resumen de qué cambió en cada tabla:

In [10]:
# tu codigo aquí
print (df_pacientes_limpio)
print (df_medicos_limpio)
print (df_turnos_limpio)


    id           nombre         dni fecha_nacimiento        ciudad  \
0    1   Martina García  28.341.992       1985-04-12  Buenos Aires   
1    2     Roberto Díaz  31.882.441       1990-11-30       Córdoba   
2    3    Luciana Pérez  35.112.008       2001-07-22           NaN   
3    4   Martina García  28.341.992       1985-04-12  Buenos Aires   
4    5       Tomás Ruiz  40.223.117       2000-03-15       Rosario   
..  ..              ...         ...              ...           ...   
61  62   Daniela Suárez  27.742.735       1961-03-08           NaN   
62  63  Ignacio Sánchez  24.037.826       1970-11-17  Bahía Blanca   
63  64   Milagros López  39.156.684       2002-11-06       Rosario   
64  65     Lucas Castro  32.279.328              NaT  Bahía Blanca   
65  66    Agustina Ruiz  36.376.846       1979-08-16      La Plata   

      obra_social  
0            OSDE  
1   Swiss Medical  
2            PAMI  
3            OSDE  
4             NaN  
..            ...  
61         Galeno  

### 5) — Análisis: responder las preguntas del director

Con los datos limpios, respondan estas cuatro preguntas. Para cada una, combinen SQL (para traer los datos necesarios) con Pandas (para calcular o presentar el resultado):

**Pregunta 1:** ¿Cuántos turnos realizó cada médico? Mostrá el resultado ordenado de mayor a menor.


**Pregunta 2:** ¿Cuál es el costo total facturado por especialidad médica? Considerá solo los turnos con estado `'realizado'`.

**Pregunta 3:** ¿Qué obra social tiene más pacientes registrados? ¿Y cuántos pacientes no tienen obra social?

**Pregunta 4:** ¿Cuántos turnos fueron cancelados o marcados como `'ausente'`? ¿Qué médico tuvo más cancelaciones?


In [12]:
# Pregunta 1

consulta = """
SELECT
    m.nombre,
    COUNT(*) AS cantidad_turnos
FROM turnos t
JOIN medicos m
ON t.medico_id = m.id
GROUP BY m.nombre
ORDER BY cantidad_turnos DESC;
"""

turnos_por_medico = pd.read_sql(consulta, conexion)

turnos_por_medico


,nombre,cantidad_turnos
0,Dra. Beatriz Sosa,34
1,Dr. Sebastián Ríos,31
2,Dra. Carmen Aguirre,28
3,Dr. Pablo Méndez,18
4,Dr. Nicolás Vera,18
5,Dra. Silvina Vega,13
6,Dra. Laura Giménez,13
7,Dra. Gabriela Luna,11
8,Dra. Andrea Blanco,11
9,Dr. Ezequiel Bravo,10


### 6) Reporte final

Escriban un bloque de código que imprima un reporte con los resultados de los cuatro puntos anteriores, en formato legible. No es necesario que sea elaborado — lo importante es que quede claro qué encontraron.

In [ ]:
print("=" * 50)
print("REPORTE CLÍNICA MEDICARE")
print("=" * 50)

# tu reporte acá


→ [Ir a la solución](./solucion.md)